# ERGT Colab Phase 3 Repair: Detached real_d

Run this notebook on Google Colab with `Runtime > Change runtime type > GPU`.

This repair run tests whether the failed seed-2027 `real_d alpha=0.2` result was caused by unstable gradient feedback through `W` and `D`. It runs `real_d` with `gradient_mode=detached_d` at alpha `0.2` and `0.1`, then compares against the tracked seed-2027 baseline and random controls.

In [ ]:
REPO_URL = "https://github.com/jalaljafari2009/ERGT.git"
BRANCH = "main"
PROJECT_DIR = "/content/ERGT"

DATA_DIR = "data/processed/wikitext2_gpt2_ctx256"
DEVICE = "cuda"

BASELINE_RESULT = "runs/phase0_baseline/confirm_seed_wikitext2/baseline_results.json"
FAILED_REAL_D = "runs/phase3_geo_attention/confirm_seed/real_d_alpha_0_2/metrics.json"
RANDOM_D_ALPHA_0_2 = "runs/phase3_geo_attention/confirm_seed/random_d_alpha_0_2/metrics.json"
RANDOM_D_ALPHA_0_1 = "runs/phase3_geo_attention/confirm_seed/random_d_alpha_0_1/metrics.json"
OUTPUT_DIR = "runs/phase3_geo_attention/phase3_repair"

RUN_REPAIR = True
RUN_COMPARISON = True
FORCE_RERUN = False

REPAIR_CONFIGS = {
    "real_d_alpha_0_2_detached": (
        "configs/ergt_v1/phase3_repair/real_d_alpha_0_2_detached_seed2027.json"
    ),
    "real_d_alpha_0_1_detached": (
        "configs/ergt_v1/phase3_repair/real_d_alpha_0_1_detached_seed2027.json"
    ),
}

REPAIR_RESULTS = {
    "real_d_alpha_0_2_detached": (
        "runs/phase3_geo_attention/phase3_repair/"
        "real_d_alpha_0_2_detached/metrics.json"
    ),
    "real_d_alpha_0_1_detached": (
        "runs/phase3_geo_attention/phase3_repair/"
        "real_d_alpha_0_1_detached/metrics.json"
    ),
}

## 1. Runtime probe

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import torch

print("cwd:", os.getcwd())
print("python:", sys.version)
print("platform:", platform.platform())
print("COLAB_RELEASE_TAG:", os.environ.get("COLAB_RELEASE_TAG"))
print("COLAB_GPU:", os.environ.get("COLAB_GPU"))

nvidia_smi = shutil.which("nvidia-smi")
print("nvidia-smi:", nvidia_smi)
if nvidia_smi:
    subprocess.run([nvidia_smi], check=False)

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("cuda_device_count:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Set Colab hardware accelerator to GPU.")
print("cuda_device_name:", torch.cuda.get_device_name(0))

## 2. Clone/update repo and install dependencies

In [ ]:
project_path = Path(PROJECT_DIR)
if project_path.exists():
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", PROJECT_DIR, "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
repo_head = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
)
print("repo:", repo_head.strip())
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,data,viz]"],
    check=True,
)

## 3. Prepare WikiText-2

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/prepare_wikitext2.py",
        "--output-dir",
        DATA_DIR,
        "--tokenizer",
        "gpt2",
        "--context-length",
        "256",
    ],
    cwd=PROJECT_DIR,
    check=True,
)

## 4. Run repair candidates

In [ ]:
def run_command(command):
    print("\n$", " ".join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)


required_reference_results = [
    BASELINE_RESULT,
    FAILED_REAL_D,
    RANDOM_D_ALPHA_0_2,
    RANDOM_D_ALPHA_0_1,
]
for path in required_reference_results:
    if not (Path(PROJECT_DIR) / path).exists():
        raise FileNotFoundError(f"Missing tracked reference result: {path}")

if RUN_REPAIR:
    for label, config_path in REPAIR_CONFIGS.items():
        result_path = Path(PROJECT_DIR) / REPAIR_RESULTS[label]
        if result_path.exists() and not FORCE_RERUN:
            print("Skipping existing repair run:", label, result_path)
            continue
        print("\n=== Phase 3 repair candidate:", label, "===")
        run_command(
            [
                sys.executable,
                "experiments/train_ergt_v1.py",
                "--config",
                config_path,
                "--data-dir",
                DATA_DIR,
                "--device",
                DEVICE,
            ]
        )
else:
    print("RUN_REPAIR is False; no repair run executed.")

for label, path in REPAIR_RESULTS.items():
    if not (Path(PROJECT_DIR) / path).exists():
        raise FileNotFoundError(f"Missing repair result for {label}: {path}")

## 5. Compare repair candidates

In [ ]:
if RUN_COMPARISON:
    command = [
        sys.executable,
        "experiments/compare_phase3_repair.py",
        "--baseline",
        BASELINE_RESULT,
        "--failed-real-d",
        FAILED_REAL_D,
        "--random-d-matched",
        RANDOM_D_ALPHA_0_2,
        "--random-d-best-control",
        RANDOM_D_ALPHA_0_1,
        "--output-dir",
        OUTPUT_DIR,
    ]
    for label, path in REPAIR_RESULTS.items():
        command.extend(["--candidate", f"{label}:{path}"])
    run_command(command)
else:
    print("RUN_COMPARISON is False; no repair comparison generated.")

## 6. Print repair report

In [ ]:
report_path = Path(PROJECT_DIR) / OUTPUT_DIR / "phase3_repair_results.json"
with report_path.open("r", encoding="utf-8") as handle:
    report = json.load(handle)

print(json.dumps(report, indent=2, sort_keys=True)[:12000])

## 7. Archive light outputs

In [ ]:
light_root = Path("/content/ergt_phase3_repair_detached_light")
if light_root.exists():
    shutil.rmtree(light_root)

include_roots = [OUTPUT_DIR]

for include_root in include_roots:
    source_dir = Path(PROJECT_DIR) / include_root
    target_dir = light_root / include_root
    target_dir.mkdir(parents=True, exist_ok=True)
    for path in source_dir.rglob("*"):
        if not path.is_file() or path.suffix not in {".json", ".jsonl"}:
            continue
        relative = path.relative_to(source_dir)
        destination = target_dir / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, destination)
        print("included:", destination.relative_to(light_root))

light_archive = shutil.make_archive(
    "/content/ergt_phase3_repair_detached_light", "zip", light_root
)
print("Light archive ready:", light_archive)